# Data Collection & Exploration

Explore LIBERO dataset: shapes, trajectories, action distributions.

In [ ]:
# Cell 1: Explore the LIBERO dataset
import sys
sys.path.insert(0, "/kaggle/working/vlm-vla/src")
from vlm_vla.data_utils import inspect_dataset

info = inspect_dataset("HuggingFaceVLA/libero")
for k, v in info.items():
    print(f"{k}: {v}")

In [ ]:
# Cell 2: Visualize a sample episode
import matplotlib.pyplot as plt
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset("HuggingFaceVLA/libero")

# Show 8 frames from episode 0
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
ep_indices = [i for i in range(len(ds)) if ds[i]["episode_index"] == 0]
step = max(1, len(ep_indices) // 8)
for idx, ax in enumerate(axes.flat):
    frame_idx = ep_indices[min(idx * step, len(ep_indices) - 1)]
    sample = ds[frame_idx]
    # Image key may vary — adapt based on inspect_dataset output
    img_key = [k for k in sample if "image" in k and hasattr(sample[k], "shape") and len(sample[k].shape) == 3][0]
    img = sample[img_key].permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(f"step {idx * step}")
    ax.axis("off")
plt.suptitle("Episode 0 trajectory")
plt.tight_layout()
plt.savefig("episode_0_viz.png", dpi=100)
plt.show()

In [ ]:
# Cell 3: Analyze action distributions
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset("HuggingFaceVLA/libero")

# Sample 1000 actions
actions = []
for i in range(0, min(1000, len(ds))):
    actions.append(ds[i]["action"].numpy())
actions = np.stack(actions)

print(f"Action shape: {actions.shape}")
print(f"Action range: [{actions.min():.3f}, {actions.max():.3f}]")
print(f"Action mean: {actions.mean(axis=0)}")
print(f"Action std:  {actions.std(axis=0)}")

# Plot per-dimension distributions
fig, axes = plt.subplots(1, actions.shape[1], figsize=(3 * actions.shape[1], 3))
labels = ["x", "y", "z", "rx", "ry", "rz", "grip"]
for d in range(min(actions.shape[1], len(labels))):
    axes[d].hist(actions[:, d], bins=50)
    axes[d].set_title(labels[d] if d < len(labels) else f"dim{d}")
plt.tight_layout()
plt.savefig("action_distribution.png", dpi=100)
plt.show()